# Proxy、Reflect 与元编程

学习目标：能用 Proxy 拦截对象操作、用 Reflect 保留默认语义，并识别代理不变量和接收者限制。

前置知识：属性描述符、访问器、原型、this、Symbol、类私有字段和异常。

适用版本：ECMAScript 2025、Node.js 24.11.0；.mjs 使用 ES 模块。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/javascript。

配套脚本：位于 scripts/28-metaprogramming/。

1. [intercept.mjs](scripts/28-metaprogramming/intercept.mjs)：读取、赋值和 receiver。
2. [descriptors.mjs](scripts/28-metaprogramming/descriptors.mjs)：枚举规则、定义结果与不变量。
3. [invariant-error.mjs](scripts/28-metaprogramming/invariant-error.mjs)：独立的 get 违约反例。
4. [revocable.mjs](scripts/28-metaprogramming/revocable.mjs)：撤销边界。
5. [protocols.mjs](scripts/28-metaprogramming/protocols.mjs)：四种 Symbol 协议的最小实现。
6. [receivers.mjs](scripts/28-metaprogramming/receivers.mjs)：私有字段和 Map 的透明性限制。

Step 1：运行拦截与 receiver 示例。

```bash
node scripts/28-metaprogramming/intercept.mjs
```

Step 2：运行描述符与枚举示例。

```bash
node scripts/28-metaprogramming/descriptors.mjs
```

Step 3：单独运行预期失败的不变量示例。

```bash
node scripts/28-metaprogramming/invariant-error.mjs
```

Step 4：运行撤销示例。

```bash
node scripts/28-metaprogramming/revocable.mjs
```

Step 5：运行 Symbol 协议。

```bash
node scripts/28-metaprogramming/protocols.mjs
```

Step 6：运行接收者限制示例。

```bash
node scripts/28-metaprogramming/receivers.mjs
```

## 1 代理拦截与 Reflect 转发

元编程（metaprogramming）指程序对语言层面的对象行为进行控制。Proxy 由目标 target 和处理器 handler 构成，get、set 等 trap（拦截函数）对应对象内部操作；未提供的 trap 通常转发给目标。代理是另一个对象，不会把目标本身变成代理，直接访问目标也不会经过拦截。

Reflect 提供与内部操作对应的方法，适合在 trap 中完成默认操作。Reflect.get 的 receiver 决定 getter 中的 this；Reflect.set 同样需要考虑接收者。简单写 target[key] 会改变接收者语义。set trap 必须给出表示成功与否的结果；在严格模式下拒绝赋值可导致 TypeError。

本例只给 count 增加本例约定的非负整数校验。Reflect 操作不绕过代理，也可能继续触发其他 trap，因此应避免在 get 内再次读取同一代理属性造成递归。

配套 [intercept.mjs](scripts/28-metaprogramming/intercept.mjs)：

```javascript
import assert from "node:assert/strict";
const events = [];
const target = { count: 2, get doubled() { return this.count * 2; } };
const proxy = new Proxy(target, {
  get(object, key, receiver) {
    events.push(`get:${String(key)}`);
    // 连同 receiver 一起转发，让 getter 中的 this 仍指向这次访问的接收者。
    return Reflect.get(object, key, receiver);
  },
  set(object, key, value, receiver) {
    // 这个分支就是 set 拦截的用途：只限制 count，其他属性照常转发。
    if (key === "count" && (!Number.isInteger(value) || value < 0)) {
      throw new RangeError("count must be nonnegative integer");
    }
    return Reflect.set(object, key, value, receiver);
  },
});
// 写入先经过 set；随后读 doubled 时，getter 又读取代理上的 count。
proxy.count = 4;
assert.equal(proxy.doubled, 8);
console.log(events.join(",")); // → get:doubled,get:count；getter 的 this 是代理
assert.throws(() => { proxy.count = -1; }, /count must be nonnegative integer/);
assert.equal(target.count, 4);
assert.notEqual(proxy, target);
console.log("receiver and validation checked"); // → receiver and validation checked
```

## 2 属性描述符与代理不变量

不变量（invariant）限制对象内部操作允许报告什么，防止代理对不可配置属性作出互相矛盾的承诺。例如目标具有不可配置且不可写的数据属性时，get 不能返回与真实值不同的结果；ownKeys 不能漏掉不可配置的自有键。

Reflect.ownKeys 返回全部自有字符串键与 Symbol 键；Object.keys 还会检查可枚举性并只返回字符串键。拦截 ownKeys 不能单独决定这些高级操作的全部结果，描述符检查仍然参与。Reflect.defineProperty 用布尔值表示定义是否成功，但不等于永远不抛错，例如参数非法或代理违约仍会抛出异常。

配套 [descriptors.mjs](scripts/28-metaprogramming/descriptors.mjs)：

```javascript
import assert from "node:assert/strict";
const marker = Symbol("marker");
const target = { visible: 1, [marker]: 2 };
Object.defineProperty(target, "hidden", { value: 3, enumerable: false, configurable: false });
const transparent = new Proxy(target, { ownKeys: Reflect.ownKeys });
assert.deepEqual(Object.keys(transparent), ["visible"]);
assert.equal(Reflect.ownKeys(transparent).length, 3);
assert.equal(Reflect.defineProperty(target, "hidden", { value: 4 }), false);
const hiding = new Proxy(target, { ownKeys() { return ["visible", marker]; } });
assert.throws(() => Reflect.ownKeys(hiding), TypeError);
console.log("keys descriptors invariants checked"); // → keys descriptors invariants checked
```

配套 [invariant-error.mjs](scripts/28-metaprogramming/invariant-error.mjs)：

```javascript
const target = Object.freeze({ code: 10 });
const invalid = new Proxy(target, { get() { return 99; } });
console.log(invalid.code); // → TypeError，非零退出：不可配置且不可写的 code 不能被报告成 99
```

## 3 可撤销代理

Proxy.revocable 返回 proxy 和 revoke。调用 revoke 后，后续需要代理内部操作的读取、赋值或函数调用会抛 TypeError；已取得的普通值不被追溯撤回。撤销可用来限定某个对象访问窗口，但如果原目标已泄露，撤销代理无法撤销对原目标的直接访问。

重复调用 revoke 无额外作用。撤销不负责释放目标持有的文件或监听器；这些资源仍须按其生命周期明确清理。

配套 [revocable.mjs](scripts/28-metaprogramming/revocable.mjs)：

```javascript
import assert from "node:assert/strict";
const target = { status: "open" };
const { proxy, revoke } = Proxy.revocable(target, {});
const value = proxy.status;
revoke();
revoke();
assert.throws(() => proxy.status, TypeError);
assert.equal(value, "open");
assert.equal(target.status, "open");
console.log("revoked proxy retained value", value); // → revoked proxy retained value open
```

## 4 内置 Symbol 协议

内置 Symbol 是语言约定的属性键，让对象参与某个操作协议；它们不是任意名称的“魔法字符串”。本章侧重协议定制，与迭代器章节的遍历过程相互补充。

| 名称 | 中文名称／含义 | 对应操作 |
| --- | --- | --- |
| Symbol.iterator | 同步迭代入口 | for...of、展开等获取迭代器 |
| Symbol.toPrimitive | 原始值转换入口 | 接收 number、string 或 default 提示 |
| Symbol.toStringTag | 对象标签 | Object.prototype.toString 的标签部分 |
| Symbol.hasInstance | 实例检查入口 | instanceof 的检查行为 |

toPrimitive 必须返回原始值，否则转换会抛 TypeError；toStringTag 是展示标签，不是可靠类型认证。自定义 hasInstance 同样会改变 instanceof 的含义，因此不能用它替代业务数据校验。

配套 [protocols.mjs](scripts/28-metaprogramming/protocols.mjs)：

```javascript
import assert from "node:assert/strict";
const range = {
  *[Symbol.iterator]() { yield 2; yield 3; },
  [Symbol.toStringTag]: "LessonRange",
};
assert.deepEqual([...range], [2, 3]);
console.log(Object.prototype.toString.call(range)); // → [object LessonRange]
const measure = {
  [Symbol.toPrimitive](hint) { return hint === "string" ? "12 cm" : 12; },
};
console.log(String(measure), +measure); // → 12 cm 12
class Even {
  static [Symbol.hasInstance](value) { return Number.isInteger(value) && value % 2 === 0; }
}
assert.equal(4 instanceof Even, true);
assert.equal(3 instanceof Even, false);
console.log("protocols checked"); // → protocols checked
```

## 5 私有字段与内置方法的接收者

空 handler 不保证代理与目标在所有操作上完全透明。类私有字段检查的是接收者是否具有该私有元素；用代理调用目标方法时，this 通常是代理，它没有目标的私有字段。Map 等内置对象的方法也检查所需内部槽，代理不会自动获得目标内部槽。

对已知方法，可显式绑定目标或暴露包装函数；这会让方法以目标为 this 执行，方法内部访问也绕过代理。不能把“所有函数都 bind(target)”当作通用透明代理方案：方法身份、访问器、继承和不变量都需要单独考虑。

配套 [receivers.mjs](scripts/28-metaprogramming/receivers.mjs)：

```javascript
import assert from "node:assert/strict";
class Vault {
  #value = 7;
  read() { return this.#value; }
}
const vault = new Vault();
const proxy = new Proxy(vault, {});
assert.throws(() => proxy.read(), TypeError);
const read = vault.read.bind(vault);
assert.equal(read(), 7);

const map = new Map([["key", 9]]);
const wrappedMap = new Proxy(map, {});
assert.throws(() => wrappedMap.get("key"), TypeError);
assert.throws(() => wrappedMap.size, TypeError);
const access = { get: (key) => map.get(key) };
assert.equal(access.get("key"), 9);
console.log("private brand and Map receiver checked"); // → private brand and Map receiver checked
```

## 本章小结

- Proxy 拦截对象操作，Reflect 帮助保留接收者与默认语义。
- 描述符和内部不变量约束代理，不是 trap 返回什么就一定成立。
- Symbol 协议可定制语言操作，私有字段和内置内部槽仍有独立约束。

## 练习

1. 给 count 赋小数，核对 RangeError；再给它赋零，核对 doubled 为 0。
2. 将 hidden 改为可配置属性后再用 ownKeys 隐藏它；标准：在目标仍可扩展时操作可以成功，解释条件变化。
3. 从代理中取得一个返回普通值的结果后撤销代理；标准：旧值仍可用，新属性访问失败，不能把撤销理解为抹除所有副本。

## 参考与引用来源

- TC39（ECMA-262 第 16 版）：[§28 Reflect 与 Proxy.revocable](https://tc39.es/ecma262/2025/multipage/reflection.html)、[§10.5 Proxy 内部操作](https://tc39.es/ecma262/2025/multipage/ordinary-and-exotic-objects-behaviours.html#sec-proxy-object-internal-methods-and-internal-slots)，特别是 Get、Set、OwnPropertyKeys 与 ValidateNonRevokedProxy；[§6.1.5.1 内置 Symbol](https://tc39.es/ecma262/2025/multipage/ecmascript-data-types-and-values.html#sec-well-known-symbols)、[§7.3 私有元素操作](https://tc39.es/ecma262/2025/multipage/abstract-operations.html#sec-privateget)、[§24.1 Map 方法](https://tc39.es/ecma262/2025/multipage/keyed-collections.html#sec-properties-of-the-map-prototype-object)：协议、私有字段与内置内部槽。